---
title: "1D Forward Simulation of Magnetotelluric Data for a Single Sounding"
authors:
  - id: devincowan
---

```{admonition} Introductory notebook
:class: hint
This tutorial teaches basic functionality within SimPEG and is a good entry point for new users.
```

```{admonition} Light-weight notebook
:class: hint
This tutorial requires minimal computational resources and can be executed quickly in the background while other computer processes are running.
```

**Keywords:** NSEM, forward simulation, 1D sounding.

</br>

**Summary:** In this tutorial, we present the fundamentals of simulating 1D NSEM data in SimPEG. We use the module [simpeg.electromagnetics.natural_source](xref:simpeg#simpeg.electromagnetics.natural_source) to simulate magnetotelluric data for a 1D sounding. We demonstrate two approaches. First, the [Simulation1DRecursive](xref:simpeg#simpeg.electromagnetics.frequency_domain.simulation_1d.Simulation1DRecursive) class is used to simulate magnetotelluric data using a recursive analytic solution. Next, the [Simulation1DElectricField](xref:simpeg#simpeg.electromagnetics.frequency_domain.simulation.Simulation1DElectricField) class is used to simulate magnetotelluric data using a mimetic finite volume approach.

**Learning Objectives:**

- The simulation of 1D NSEM data with SimPEG.
- The data convention SimPEG uses for NSEM.
- Understanding the way in which NSEM surveys are created in SimPEG, which includes:
    - Defining receivers
    - Defining natural electromagnetic sources
    - Organizing sources and receivers into a survey object

## Importing Modules

Here, we import all of the functionality required to run the notebook for the tutorial exercise. All of the functionality specific to NSEM is imported from [simpeg.electromagnetics.natural_source](xref:simpeg#simpeg.electromagnetics.natural_source). To generate the mesh, we use the [discretize](https://discretize.simpeg.xyz/en/main) package.
We also import some useful utility functions from [simpeg.utils](xref:simpeg#simpeg.utils).

In [1]:
# SimPEG functionality
import simpeg.electromagnetics.natural_source as nsem
from simpeg import maps
from simpeg.utils import plot_1d_layer_model, get_default_solver

# discretize functionality
from discretize import TensorMesh

# Common Python functionality
import os
import numpy as np
from scipy.constants import mu_0
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams.update({"font.size": 14})

write_output = False  # Optional

## Recursive 1D Solution

Here, we use the [Simulation1DRecursive](xref:simpeg#simpeg.electromagnetics.frequency_domain.simulation_1d.Simulation1DRecursive) class to simulate magnetotelluric data using a recursive analytic solution. This approach is easiest and most effective for simulating NSEM sounding data over a simple 1D layered Earth.

### Defining the Survey

To generate an NSEM survey, the user must create and connect three types of objects:

- **receivers:** which define the locations of the NSEM fields being measured and the data type (e.g. impedance, tipper). The recursive 1D solution is used explicitly to compute 1D sounding data corresponding to the $Z_{xy}$ and $Z_{yx}$ impedances. Receivers for simulating these data are defined using the [Impedance](xref:simpeg#simpeg.electromagnetics.natural_source.receivers.Impedance) class.
- **sources:** which defines the frequencies of the incident planewave source. For a 1D problem geometry, we use the [BasePlanewave](xref:simpeg#simpeg.electromagnetics.static.natural_source.source.Planewave) class.
- **survey:** which uses the [survey](xref:simpeg#simpeg.electromagnetics.natural_source.Survey) class to store and organizes all of the sources and receivers.

**For this tutorial**, we simulate all of the data types corresponding to the $Z_{xy}$ impedance at 11 logarithmically spaced frequencies between 0 Hz and 100 Hz. This includes the real component of the impedance (V/A), the imaginary component (V/A), the apparent resistivity ($\Omega m$) and the phase (deg).

In [2]:
# Source properties
frequencies = np.logspace(0, 2, 11)  # frequencies in Hz

# Receiver properties.
# Since the recursive 1D solution simulates impedance data at the Earth's surface,
# the location of the field measurements is set with a dummy value. By only setting
# the location of the electric field measurement, we assume the magnetic field
# measurement is at the same location.
location_recursive = 0.  # float or (1,) numpy.ndarray
orientation = 'xy'  # Only 'xy' or 'yx' for 1D simulations
components = ["real", "imag", "apparent_resistivity", "phase"]

The easiest way to define a 1D NSEM survey is to loop over all planewave sources. A new source object must be created for each frequency. For each source, we define and assign the associated receivers. A different receiver object must be created for each data type being simulated.

In [3]:
source_list_recursive = []  # create empty list for all source objects

# loop over all sources (frequencies)
for freq in frequencies:
    
    # Define receivers for Re[Zxy], Im[Zxy], app_res and phase.
    receiver_list_recursive = []
    for comp in components:
        receiver_list_recursive.append(
            nsem.receivers.Impedance(
                locations_e=location_recursive,
                orientation=orientation,
                component=comp,
            )
        )

    # Define the planewave source and connect receivers
    source_list_recursive.append(
        nsem.sources.BasePlanewave(
            receiver_list=receiver_list_recursive,
            frequency=freq,
        )
    )

# Define the NSEM survey
survey_recursive = nsem.survey.Survey(source_list_recursive)

AttributeError: module 'simpeg.electromagnetics.natural_source.sources' has no attribute 'BasePlanewave'

### Defining a 1D Layered Earth

In SimPEG, a 1D layered Earth is defined by the set of layer thicknesses and the physical properties for each layer. If we have N layers, we define N physical property values and N-1 layer thicknesses. The lowest layer is assumed to extend to infinity. In the case of a halfspace, the layer thicknesses would be an empty array.

:::{Warning} Ordering of arrays.
Thicknesses and physical property values for the *Simulation1DRecursive* class are defined from the **bottom layer up!** So the first entry in the array defining the layer conductivities/resistivities would be the electrical conductivity/resistivity of the infinite halfspace. However, the first entry in the array defining the thicknesses is the thickness of the layer above the infinite halfspace.
:::

**For the tutorial,** we have a 3-layered Earth. The top layer has a thickness of 150 m and an electrical resistivity of 10 $\Omega m$. The middle layer has a thickness of 50 m and an electrical resistivity of 1 $\Omega m$. The third layer (infinite halfspace) has an electrical resistivity of 10 $\Omega m$.

In [ ]:
# REMEMBER TO DEFINE FROM BOTTOM TO TOP!!!

# Define layer thicknesses (m)
layer_thicknesses = np.array([50.0, 150.0])

# Define layer resistivities (Ohmm)
layer_resistivities = np.r_[10.0, 1.0, 10.0]

Here we plot the 1D layered Earth model. To use the [plot_1d_layer_model](xref:simpeg#simpeg.plot_1d_layer_model), we need to reverse the ordering of the layer thicknesses and resistivities.

In [ ]:
fig = plt.figure(figsize=(4, 5))

layer_thicknesses_plotting = np.flipud(layer_thicknesses)
layer_resistivities_plotting = np.flipud(layer_resistivities)

ax1 = fig.add_axes([0.1, 0.1, 0.8, 0.8])
ax1 = plot_1d_layer_model(layer_thicknesses_plotting, layer_resistivities_plotting, scale="log", ax=ax1)
ax1.grid(which="both")
ax1.set_xlabel(r"Resistivity ($\Omega m$)")

plt.show()

### Models and Mappings

In SimPEG, the term 'model' is not necessarily synonymous with a set of physical property values. For example, the model may be defined as the logarithms of the physical property values, or be the parameters defining a layered Earth geometry. Models in SimPEG are 1D [numpy.ndarray](xref:numpy#numpy.ndarray) whose lengths are equal to the number of model parameters. For 1D NSEM simulations, we can characterize the Earth's electric properties according to electrical conductivity or electrical resistivity.

Classes within the ``simpeg.maps`` module are used to define the mapping that connects the model to the parameters required to run the 1D NSEM simulation; i.e. layer conductivities/resistivities. In this tutorial, the layer thicknesses are fixed in the simulation. We define the model as the layer resistivities. And the mapping from the model to the resistivities is defined using the [simpeg.maps.IdentityMap](xref:simpeg#simpeg.maps.IdentityMap) class.

In [ ]:
# Number of parameters
n_layers = len(layer_resistivities)

# Resistivity model
resistivity_model = layer_resistivities.copy()

# Mapping
resistivity_map = maps.IdentityMap(nP=3)

### Defining the Forward Simulation

In SimPEG, the physics of the forward simulation is defined by creating an instance of an appropriate simulation class. Here, we use the [Simulation1DRecursive](xref:simpeg#simpeg.electromagnetics.natural_source.Simulation1DRecursive) which simulates the data according to a recursive analytic solution for the impedance at the Earth's surface. To fully define the forward simulation, we need to connect the simulation object to:

- the survey
- the layer thicknesses
- the mapping from the model to the layer conductivities/resistivities

This is accomplished by setting each one of the aforementioned items as a property of the simulation object. We use ``rhoMap`` if the electrical properties are defined according to electrical resistivity. We use ``sigmaMap`` if the electrical properties are defined according to electrical conductivity. 

In [ ]:
simulation_recursive = nsem.Simulation1DRecursive(
    survey=survey_recursive,
    thicknesses=layer_thicknesses,
    rhoMap=resistivity_map,
)

### Predict 1D NSEM Data

Once any simulation within SimPEG has been properly constructed, simulated data for a given model vector can be computed using the [dpred](xref:simpeg#simpeg.simulation.BaseSimulation.dpred) method. For 1D surveys consisting of multiple source frequencies and multiple receivers per source, the predicted data vector is organized:

- by source (frequency)
- by receiver (data type)

In [ ]:
dpred_recursive = simulation_recursive.dpred(resistivity_model)

:::{warning} Warning: SimPEG NSEM data convention!
SimPEG uses a universally right-handed coordinate system with X = Easting, Y = Northing and Z positive upward, and a $+i\omega t$ Fourier convention. Consequently, the $Z_{xy}$ impedance (plotted below) lives in the lower-left quadrant of the complex plane while $Z_{yx}$ impedance lives in the upper-right quadrant.
:::

In [ ]:
fig = plt.figure(figsize=(16, 3.5))

ax1 = fig.add_axes([0.1, 0.05, 0.2, 0.9])
ax2 = fig.add_axes([0.4, 0.05, 0.2, 0.9])
ax3 = fig.add_axes([0.7, 0.05, 0.2, 0.9])

ax1.semilogx(frequencies, dpred_recursive[0::4], "b-", lw=2)
ax1.semilogx(frequencies, dpred_recursive[1::4], "b--", lw=2)
ax1.grid(which='both')
ax1.set_xlabel("Frequency (Hz)")
ax1.set_ylabel("$Z_{xy}$ (V/A)")
ax1.set_title("Impedance")
ax1.legend(["Re[Zxy]", "Im[Zxy]"])

ax2.loglog(frequencies, dpred_recursive[2::4], "r", lw=2)
ax2.grid(which='both')
ax2.set_xlabel("Frequency (Hz)")
ax2.set_ylabel("$\\rho_{xy}$ ($\\Omega m$)")
ax2.set_title("Apparent Resistivity")

ax3.semilogx(frequencies, dpred_recursive[3::4], "g", lw=2)
ax3.grid(which='both')
ax3.set_xlabel("Frequency (Hz)")
ax3.set_ylabel("$\\phi$ (deg)")
ax3.set_title("Phase")

plt.show()

## 1D Finite Volume Simulation

Here, we use the [Simulation1DElectricField](xref:simpeg#simpeg.electromagnetics.frequency_domain.simulation.Simulation1DElectricField) class to simulate magnetotelluric data using a mimetic finite volume approach.

### Defining the Survey

Once again, the NSEM surveys for a 1D problem geometry require that we define:

- **receivers:** which can only be used to simulate data for the $Z_{xy}$ and $Z_{yx}$ impedances for 1D problems. Once again, we use the [Impedance](xref:simpeg#simpeg.electromagnetics.natural_source.receivers.Impedance) class.
- **sources:** which defines the frequencies of the incident planewave source. To solve the problem using a finite volume approach, we must use the [PlanewaveXYPrimary](xref:simpeg#simpeg.electromagnetics.static.natural_source.source.PlanewaveXYPrimary) class.
- **survey:** which uses the [survey](xref:simpeg#simpeg.electromagnetics.natural_source.Survey) class to store and organizes all of the sources and receivers.

**For this tutorial**, we simulate the real and imaginary components of $Z_{xy}$ and $Z_{yx}$ at 11 logarithmically spaced frequencies between 0 Hz and 100 Hz. Here, the location of the receivers (i.e. its elevation) matters!!! We will demonstrate this by setting the elevation to 100 m.

In [ ]:
# Source properties
frequencies = np.logspace(0, 2, 11)  # frequencies in Hz

# Receiver properties
receiver_elevation = 100.  # float or (1,) numpy.ndarray
orientations = ["xy", "yx"]  # 'xy' or 'yx' for 1D simulations
components = ["real", "imag"]

In [ ]:
source_list_voxel = []  # create empty list for source objects

# loop over all sources (frequencies)
for freq in frequencies:

    receiver_list_voxel = []
    for orient in orientations:
        for comp in components:
            receiver_list_voxel.append(
                nsem.receivers.Impedance(
                    locations_e=receiver_elevation,
                    orientation=orient,
                    component=comp,
                )
            )

    source_list_voxel.append(
        nsem.sources.PlanewaveXYPrimary(
            receiver_list=receiver_list_voxel,
            frequency=freq,
        )
    )

survey_voxel = nsem.survey.Survey(source_list_voxel)

### Defining a 1D Layered Earth

The mimetic finite volume approach provides a discrete solution of the electromagnetic fields on a 1D mesh (voxel grid). For this, we define a 1D [tensor mesh](xref:discretize#discretize.TensorMesh). Meshes are designed using the [discretize package](https://discretize.simpeg.xyz).

The accuracy of the solution, and our predicted data, depends on how we define the 1D mesh. We require sufficiently fine cells to accurately simulate the physics at the highest frequencies. And the extent of the mesh must be far enough to respect the boundary conditions for the partial differential equation being solved.

For the largest conductivity in our model and highest frequency, we compute the minimum skin depth:

$$
d_{min} \approx 500 \sqrt{\dfrac{1}{\sigma_{max} \, f_{max}}}
$$

The minimum cell width is some fraction of the minimum skin depth. Next, we use the minimum conductivity and lowest frequency compute the maximum skin depth:

$$
d_{max} \approx 500 \sqrt{\dfrac{1}{\sigma_{min} \, f_{min}}}
$$

**For this tutorial,** we simulate the data for the same 3-layered Earth as in the previous example. In this case, the surface is defined at an elevation of 100 m. We must take this into account when defining the elevations for the interfaces between layers. Instead of resistivities, we also define the electrical properties of the layers as conductivities.

In [ ]:
top_elevation = -50.  # receiver elevation - top layer thickness
bottom_elevation = -100.  # receiver elevation - top and middle layer thicknesses  

# Define unit conductivities (S/m)
air_conductivity = 1e-8
host_conductivity = 0.1
layer_conductivity = 1.0

In [ ]:
# minimum skin depth
d_min = 500.0 / np.sqrt(layer_conductivity * frequencies.max())
print("MINIMUM SKIN DEPTH: {} m".format(d_min))

# maximum skin depth
d_max = 500.0 / np.sqrt(host_conductivity * frequencies.min())
print("MAXIMUM SKIN DEPTH: {} m".format(d_max))

To create the mesh, we start by defining a "core mesh" region where the minimum cell size is used within the smallest skin depth. Outside of this region, we pad out with exponentially increasing cell sizes until the padding region is 2-3 times the largest skin depth.

**For our tutorial:** we set the minimum cell size to be 5 m. We start by defining a tensor mesh whose center is at 0 m. We then shift the mesh "up" by the height of our receiver, then "down" by 300 m. We do this for two reasons:
1. Most important, our receiver lies directly on a mesh node, which we will define as the Earth's surface.
2. Our core mesh region is mostly cells below the Earth's surface.

In [ ]:
# Generate tensor mesh with top at z = 0 m
dh = 5.
hx = [(dh, 50, -1.1), (dh, 100), (dh, 50, 1.1)]
mesh = TensorMesh([hx], "C")

# Shift origin
mesh.origin += receiver_elevation - 300.

# Mesh properties
print(mesh)

For voxel models, we must assign a conductivity (or resistivity) value to each mesh cell. **Important:** note that the conductivity value for air cells is 1e-8 S/m and not 0!!! This is required for the linear system of discrete equations that is solved remains well-conditioned. If one were to reproduce the result using an electrical resistivity model, air cells would be assigned values of 1e8 $\Omega m$ instead of $\infty$ for the same reason.

In [ ]:
conductivity_model = air_conductivity * np.ones(mesh.n_cells)
conductivity_model[mesh.cell_centers < receiver_elevation] = host_conductivity
inds = (mesh.cell_centers < top_elevation) & (mesh.cell_centers > bottom_elevation)
conductivity_model[inds] = layer_conductivity

### Define the Mapping

Here, our model represents an electrical conductivity value for each mesh cell. To define the mapping from the model to the mesh, we simply use the [simpeg.maps.IdentityMap](xref:simpeg#simpeg.maps.IdentityMap) class.

In [ ]:
conductivity_map = maps.IdentityMap(mesh)

### Defining the Forward Simulation

In SimPEG, the physics of the forward simulation is defined by creating an instance of an appropriate simulation class. Here, we use the [Simulation1DElectricField](xref:simpeg#simpeg.electromagnetics.natural_source.Simulation1DElectricField) which simulates the data according to 1D finite volume solution for the total electric field on mesh nodes. To fully define the forward simulation, we need to connect the simulation object to:

- the survey
- the mesh
- the mapping from the model to the conductivities/resistivities
- the numerical solver used to solve the discrete system at each frequency

This is accomplished by setting each one of the aforementioned items as a property of the simulation object. We use ``sigmaMap`` if the electrical properties are defined according to electrical conductivity.  We use ``rhoMap`` if the electrical properties are defined according to electrical resistivity. We use the ``get_default_solver`` utility function to provide a suitable solver.

In [ ]:
simulation_voxel = nsem.simulation.Simulation1DElectricField(
    survey=survey_voxel,
    mesh=mesh,
    sigmaMap=conductivity_map,
    solver=get_default_solver()
)

## Predict 1D NSEM Data

In [ ]:
dpred_voxel = simulation_voxel.dpred(conductivity_model)

:::{warning} Warning: SimPEG NSEM data convention!
SimPEG uses a universally right-handed coordinate system with X = Easting, Y = Northing and Z positive upward, and a $+i\omega t$ Fourier convention. Consequently, the $Z_{xy}$ impedance (plotted below) lives in the lower-left quadrant of the complex plane while $Z_{yx}$ impedance lives in the upper-right quadrant.
:::

In [ ]:
fig = plt.figure(figsize=(4, 5))

ax1 = fig.add_axes([0.1, 0.05, 0.85, 0.9])

ax1.semilogx(frequencies, dpred_voxel[0::4], "b-", lw=2)
ax1.semilogx(frequencies, dpred_voxel[1::4], "b--", lw=2)
ax1.semilogx(frequencies, dpred_voxel[2::4], "r-", lw=2)
ax1.semilogx(frequencies, dpred_voxel[3::4], "r--", lw=2)
ax1.grid(which='both')
ax1.set_xlabel("Frequency (Hz)")
ax1.set_ylabel("$Z_{ij}$ (V/A)")
ax1.set_title("Impedance")
ax1.legend(["Re[Zxy]", "Im[Zxy]", "Re[Zyx]", "Im[Zyx]"])

**Optional:** Export data.

In [ ]:
if write_output:
    dir_path = os.path.sep.join([".", "fwd_fdem_1d_outputs"]) + os.path.sep
    if not os.path.exists(dir_path):
        os.mkdir(dir_path)

    rng = np.random.default_rng(seed=222)
    noise = rng.normal(
        scale=0.05 * np.abs(dpred_conductivity),
        size=len(dpred_conductivity),
    )
    dpred_out = dpred_conductivity + noise

    fname = dir_path + "em1dfm_data.txt"
    np.savetxt(
        fname,
        np.c_[frequencies, dpred_out[0::2], dpred_out[1::2]],
        fmt="%.4e",
        header="FREQUENCY HZ_REAL HZ_IMAG",
    )